# Merging datasets to one df 

Merges precipitation (IMERG/PPS), NO$_2$ pulses, SMAP (linearly interpolated onto the hourly grid), soil properties, state, timezone and local time

In [12]:
import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from timezonefinder import TimezoneFinder

In [13]:
YEAR = 2024
SFX = {2024: "_2024", 2026: "", 2025: "_2025"}[YEAR]          
PSFX = {2024: "_24", 2026: "", 2025: "_25"}[YEAR]      

GRID = "/global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets"
precip_file = f"{GRID}/gridded_and_crop_filtered/PPS_03-07_0.1x0.1gridded_cropfiltered_MCD12Q1mask{SFX}.parquet"
no2_pulse_file = f"/global/scratch/users/rubypercival/TEMPO/parquet_V04/GCF_no2_pulses_mar-july_3SD_2hourpulse_all_hours_MCD12Q1mask{PSFX}_FIXED.parquet"
smap_file = f"{GRID}/gridded_and_crop_filtered/SMAP_03-07_0.1x0.1gridded_cropfiltered_MCD12Q1mask{SFX}.parquet"
soil_props_file = f"{GRID}/gridded_and_crop_filtered/soil_props_GCF_MCD12Q1mask.parquet"
states_file = "/global/scratch/users/rubypercival/US_State_Shapefile/tl_2024_us_state/tl_2024_us_state.shp"
output_dir = Path(f"{GRID}/final_datasets/lowCF_merged_all_datasets_allcols_MCD12Q1mask{SFX}.parquet")

STATES_TO_KEEP = ["Missouri", "Iowa", "Michigan", "South Dakota", "Kansas", "Nebraska", "Wisconsin", "Minnesota",
                  "North Dakota", "Illinois", "Tennessee", "Arkansas", "Kentucky", "Indiana", "Ohio"]
LAT_BANDS = np.arange(36, 49)   
merge_keys = ["datetime", "grid_lat", "grid_lon"]

for p in [precip_file, no2_pulse_file, smap_file, soil_props_file, states_file]:
    print("OK " if Path(p).exists() else "MISSING ", p)
print("output:", output_dir)

OK  /global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/PPS_03-07_0.1x0.1gridded_cropfiltered_MCD12Q1mask_2024.parquet
OK  /global/scratch/users/rubypercival/TEMPO/parquet_V04/GCF_no2_pulses_mar-july_3SD_2hourpulse_all_hours_MCD12Q1mask_24_FIXED.parquet
OK  /global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/SMAP_03-07_0.1x0.1gridded_cropfiltered_MCD12Q1mask_2024.parquet
OK  /global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/soil_props_GCF_MCD12Q1mask.parquet
OK  /global/scratch/users/rubypercival/US_State_Shapefile/tl_2024_us_state/tl_2024_us_state.shp
output: /global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/final_datasets/lowCF_merged_all_datasets_allcols_MCD12Q1mask_2024.parquet


In [14]:
soil_props_df = pd.read_parquet(soil_props_file, engine="pyarrow").rename(columns={"lat": "grid_lat", "lon": "grid_lon"})
soil_props_df["grid_lat"] = soil_props_df["grid_lat"].astype("float64").round(2)
soil_props_df["grid_lon"] = soil_props_df["grid_lon"].astype("float64").round(2)

states = (gpd.read_file(states_file)[["STUSPS", "NAME", "geometry"]]
          .rename(columns={"STUSPS": "state", "NAME": "state_name"}).to_crs(epsg=4326))
tf = TimezoneFinder()
print(f"soil props: {len(soil_props_df):,} cells | states: {len(states)}")

soil props: 10,995 cells | states: 56


In [15]:
def read_band(path, lo, hi, rename=None):
    """read only rows with lo <= grid_lat < hi straight from the parquet file"""
    d = pd.read_parquet(path, engine="pyarrow", filters=[("grid_lat", ">=", lo - 1e-4), ("grid_lat", "<", hi - 1e-4)])
    if rename:
        d = d.rename(columns=rename)
    d["grid_lat"] = d["grid_lat"].astype("float64").round(2)
    d["grid_lon"] = d["grid_lon"].astype("float64").round(2)
    d = d[(d["grid_lat"] >= lo) & (d["grid_lat"] < hi)]
    d["datetime"] = pd.to_datetime(d["datetime"]).astype("datetime64[ns]")   # same precision in every file (ms vs us breaks merge_asof)
    return d

def cell_info(cells):
    """state, state_name, timezone for each unique (grid_lat, grid_lon)"""
    g = gpd.GeoDataFrame(cells.copy(), geometry=gpd.points_from_xy(cells["grid_lon"], cells["grid_lat"]), crs="EPSG:4326")
    s = gpd.sjoin(g, states, how="left", predicate="within")[["grid_lat", "grid_lon", "state", "state_name"]]
    s = s.drop_duplicates(["grid_lat", "grid_lon"])
    s["timezone"] = [tf.timezone_at(lat=a, lng=b) or "UTC" for a, b in zip(s["grid_lat"], s["grid_lon"])]
    return s

SMAP_MAX_GAP_H = 3   # interpolate only between SMAP samples <= 3 h apart (SMAP L4 is 3-hourly)

def smap_to_hourly(smap, targets):
    """Linearly interpolate SMAP (any timestamps, e.g. 3-hourly at ~:30) onto the hourly (cell, datetime) targets.
    Uses the nearest SMAP sample before and after each target hour (each within SMAP_MAX_GAP_H); exact matches pass through."""
    num = [c for c in smap.columns if c not in ("datetime", "grid_lat", "grid_lon") and pd.api.types.is_numeric_dtype(smap[c])]
    sm = smap[["grid_lat", "grid_lon", "datetime"] + num].sort_values("datetime").rename(columns={"datetime": "t_s"})
    tg = targets.sort_values("datetime")
    tol = pd.Timedelta(hours=SMAP_MAX_GAP_H)
    kw = dict(left_on="datetime", right_on="t_s", by=["grid_lat", "grid_lon"], tolerance=tol)
    prev = pd.merge_asof(tg, sm, direction="backward", **kw)
    nxt = pd.merge_asof(tg, sm, direction="forward", **kw)
    span = (nxt["t_s"] - prev["t_s"]).dt.total_seconds()
    w = ((prev["datetime"] - prev["t_s"]).dt.total_seconds() / span.where(span > 0)).fillna(0.0)
    out = prev[["grid_lat", "grid_lon", "datetime"]].copy()
    for c in num:
        out[c] = (prev[c] + w * (nxt[c] - prev[c])).astype("float64")
    return out

def to_local(dt_utc, tz_series):
    out = pd.Series(pd.NaT, index=dt_utc.index, dtype="datetime64[ns]")
    utc = dt_utc.dt.tz_localize("UTC")
    for tz, idx in tz_series.groupby(tz_series).groups.items():
        out.loc[idx] = utc.loc[idx].dt.tz_convert(tz).dt.tz_localize(None).astype("datetime64[ns]")
    return out

In [16]:
FRESH_START = True
STEP = 0.5
BANDS = np.arange(36, 49, STEP)

if FRESH_START and output_dir.exists():
    shutil.rmtree(output_dir) if output_dir.is_dir() else output_dir.unlink()
output_dir.mkdir(parents=True, exist_ok=True)

total_rows = total_pulses = 0
failed = []
for b in BANDS:
    lo, hi = float(b), float(b + STEP)
    out_f = output_dir / f"band_{lo:.1f}N.parquet"
    if out_f.exists():
        print(f"{lo:.1f}-{hi:.1f}N: already done, skipping", flush=True)
        continue
    try:
        precip = read_band(precip_file, lo, hi, rename={"hour": "datetime"})
        precip = precip.drop(columns=[c for c in ["date", "n_halfhours"] if c in precip.columns])
        no2 = read_band(no2_pulse_file, lo, hi)
        if "datetime" not in no2.columns and "datetime_utc" in no2.columns:
            no2 = no2.rename(columns={"datetime_utc": "datetime"})
        smap = read_band(smap_file, lo, hi, rename={"lat": "grid_lat", "lon": "grid_lon"})
        print(f"{lo:.1f}-{hi:.1f}N: read precip {len(precip):,} | no2 {len(no2):,} | smap {len(smap):,}", flush=True)

        # put SMAP on the same hourly timestamps as precip/NO2 (SMAP L4 is 3-hourly, at ~:28)
        hours = pd.concat([precip[merge_keys], no2[merge_keys]]).drop_duplicates()
        smap = smap_to_hourly(smap, hours)
        del hours; gc.collect()

        m = (precip.merge(no2, on=merge_keys, how="outer")
                   .merge(smap, on=merge_keys, how="outer", suffixes=("", "_smap")))
        del precip, no2, smap; gc.collect()
        m = m.merge(soil_props_df, on=["grid_lat", "grid_lon"], how="left")

        info = cell_info(m[["grid_lat", "grid_lon"]].drop_duplicates())
        m = m.drop(columns=[c for c in ["state", "state_name", "timezone"] if c in m.columns])
        m = m.merge(info, on=["grid_lat", "grid_lon"], how="left")
        m = m[m["state_name"].isin(STATES_TO_KEEP)]            # drop out-of-state rows BEFORE the slow local-time step
        m["datetime_local"] = to_local(m["datetime"], m["timezone"])
        m = m.sort_values(merge_keys).reset_index(drop=True)
        for c in m.columns[m.dtypes == "object"]:
            m[c] = m[c].astype(str)

        n_p = int(m["pulse_start"].astype(str).eq("True").sum()) if "pulse_start" in m.columns else 0
        if len(m):
            m.to_parquet(out_f, engine="pyarrow", index=False)
        total_rows += len(m); total_pulses += n_p
        print(f"{lo:.1f}-{hi:.1f}N: wrote {len(m):>11,} rows, {n_p:>6,} pulse starts", flush=True)
        del m, info; gc.collect()
    except Exception as e:
        failed.append(lo)
        print(f"{lo:.1f}-{hi:.1f}N: FAILED -> {type(e).__name__}: {e}", flush=True)
        gc.collect()

print(f"\nTHIS RUN: {total_rows:,} rows, {total_pulses:,} pulse starts")
print(f"band files in folder: {len(list(output_dir.glob('*.parquet')))}  ->  {output_dir}")
print("failed bands:", failed or "none")

36.0-36.5N: read precip 664,632 | no2 151,204 | smap 221,725
36.0-36.5N: wrote     356,184 rows,    279 pulse starts
36.5-37.0N: read precip 708,696 | no2 154,321 | smap 236,425
36.5-37.0N: wrote     396,576 rows,    280 pulse starts
37.0-37.5N: read precip 682,992 | no2 160,316 | smap 227,850
37.0-37.5N: wrote     668,304 rows,    416 pulse starts
37.5-38.0N: read precip 1,006,128 | no2 233,028 | smap 335,650
37.5-38.0N: wrote   1,002,456 rows,    548 pulse starts
38.0-38.5N: read precip 1,237,464 | no2 278,911 | smap 412,825
38.0-38.5N: wrote   1,215,432 rows,    572 pulse starts
38.5-39.0N: read precip 1,046,520 | no2 220,502 | smap 349,125
38.5-39.0N: wrote   1,042,848 rows,    427 pulse starts
39.0-39.5N: read precip 1,711,152 | no2 357,204 | smap 570,850
39.0-39.5N: wrote   1,663,416 rows,    713 pulse starts
39.5-40.0N: read precip 2,467,584 | no2 497,810 | smap 823,200
39.5-40.0N: wrote   2,452,896 rows,  1,146 pulse starts
40.0-40.5N: read precip 2,706,264 | no2 538,103 | smap

## Checks (reads only a few columns back)

In [17]:
assert any(output_dir.glob("*.parquet")), "no band files yet -- run the merge cell above first"
chk = pd.read_parquet(output_dir, columns=["grid_lat", "grid_lon", "datetime", "h_precip_mm", "no2_mean", "pulse_start", "WFPS_c", "state_name"])
print(f"rows: {len(chk):,} | cells: {chk[['grid_lat', 'grid_lon']].drop_duplicates().shape[0]:,}")
print(f"duplicate cell-hours: {chk.duplicated(['grid_lat', 'grid_lon', 'datetime']).sum():,}  (should be 0)")
print(f"precip coverage: {chk['h_precip_mm'].notna().mean():.1%} | mean precip: {24 * chk['h_precip_mm'].mean():.2f} mm/day (expect ~2-4)")
print(f"pulse starts: {chk['pulse_start'].astype(str).eq('True').sum():,}")
print(f"rows with NO2: {chk['no2_mean'].notna().sum():,} | rows with WFPS: {chk['WFPS_c'].notna().sum():,}")
p = chk[chk["pulse_start"].astype(str).eq("True")]
print(f"pulse rows with WFPS: {p['WFPS_c'].notna().mean():.1%}  (should be ~100%)")
print("minutes past the hour in datetime:", sorted(chk["datetime"].dt.minute.unique())[:5], " (should be [0])")
print(chk["state_name"].value_counts().head(15))
del chk; gc.collect()

rows: 38,482,560 | cells: 10,480
duplicate cell-hours: 0  (should be 0)
precip coverage: 59.0% | mean precip: 3.88 mm/day (expect ~2-4)
pulse starts: 16,583
rows with NO2: 7,146,013 | rows with WFPS: 38,461,067
pulse rows with WFPS: 100.0%  (should be ~100%)
minutes past the hour in datetime: [np.int32(0)]  (should be [0])
state_name
North Dakota    5452920
Iowa            5342760
Illinois        4784616
Minnesota       4255848
Kansas          3146904
Nebraska        3132216
South Dakota    3044088
Indiana         2489616
Missouri        2004912
Ohio            1784592
Wisconsin       1310904
Michigan        1112616
Kentucky         367200
Tennessee        128520
Arkansas         124848
Name: count, dtype: int64


0

In [18]:
import shutil
files = sorted(output_dir.glob("*"))
band = [f for f in files if f.name.startswith("band_")]
other = [f for f in files if not f.name.startswith("band_")]
print(f"{len(band)} band files, {len(other)} other files:", [f.name for f in other][:10])

old_dir = output_dir.with_name(output_dir.name.replace(".parquet", "") + "_OLD")
old_dir.mkdir(exist_ok=True)
for f in other:
    shutil.move(str(f), old_dir / f.name)
print("moved to", old_dir)

26 band files, 0 other files: []
moved to /global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/final_datasets/lowCF_merged_all_datasets_allcols_MCD12Q1mask_2024_OLD


In [19]:
import pyarrow.parquet as pq
print("R_2_1 precip_file:", precip_file)
pf = pq.ParquetFile(precip_file)
print("columns:", pf.schema_arrow.names)
print("rows:", f"{pf.metadata.num_rows:,}")

R_2_1 precip_file: /global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/PPS_03-07_0.1x0.1gridded_cropfiltered_MCD12Q1mask_2024.parquet
columns: ['grid_lat', 'grid_lon', 'hour', 'h_precip_mm', 'n_halfhours']
rows: 40,373,640
